In [1]:
import argparse
import json
import os
import re
from pathlib import Path
from typing import Dict, Optional

import pytorch_lightning as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio.functional as ta_F
import yaml
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.strategies import DDPStrategy

from src.spatial_attn_lightning import BinauralAttentionModule

/orcd/data/jhm/001/om2/rphess/projects/github.com/Auditory-Attention/phaselocknet_torch/peripheral_model.py:421: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)


In [2]:
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

torch.set_float32_matmul_precision("medium")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [3]:
class StageFeatureExtractor(nn.Module):
    """
    Wraps a model and extracts features at a chosen internal stage using a forward hook.
    """

    def __init__(self, backbone: nn.Module, target_stage: str):
        super().__init__()
        self.backbone = backbone
        self.target_stage = target_stage
        self._features = None

        # Register hook at target stage
        for name, module in self.backbone.named_modules():
            if target_stage in name:
                module.register_forward_hook(self._hook)
                break
        else:
            raise ValueError(f"Stage {target_stage} not found in backbone")

    def _hook(self, module, input, output):
        self._features = output

    def forward(self, **inputs):
        _ = self.backbone(**inputs)  # pass everything to backbone
        return self._features


class MultiClassifierModule(pl.LightningModule):
    def __init__(
        self,
        backbone: nn.Module,
        audio_transforms: nn.Module,
        target_stage: str,
        layer_idx: int,
        task_dict: Dict[str, int],
        lr_dict: Dict[str, float],
        save_dir: Optional[str] = "linear_readout_weights",
        save_outputs: bool = False,
    ):
        super().__init__()
        self.save_hyperparameters(ignore=["backbone"])
        self.task_dict = task_dict
        self.task_names = list(task_dict.keys())
        self.audio_transforms = audio_transforms
        self.automatic_optimization = False
        self.layer_idx = layer_idx

        # Feature extractor
        self.feature_extractor = StageFeatureExtractor(backbone, target_stage)
        for p in self.feature_extractor.parameters():
            p.requires_grad = False

        # Infer feature dim
        with torch.no_grad():
            dummy = torch.randn(1, 2, 110_250, device="cuda")
            feats = self.feature_extractor(mixture=dummy)
            # delete dummy
            del dummy
            feat_dim = feats.view(1, -1).shape[1]

        # Classifier heads
        self.classifier_heads = nn.ModuleDict(
            {
                task: nn.Linear(feat_dim, num_classes)
                for task, num_classes in task_dict.items()
            }
        )

        self.f0_bin_counts = {}
        self.count_f0_bins = True  # Flag to control f0 bin counting

        # Tracking best validation loss per task and running sums
        self.best_val_loss: Dict[str, float] = {
            task: float("inf") for task in self.task_names
        }
        self.val_loss_sum: Dict[str, float] = {task: 0.0 for task in self.task_names}
        self.val_total: Dict[str, int] = {task: 0 for task in self.task_names}

        # Where to save best-performing classifier head weights
        self.save_dir = Path(save_dir)
        self.save_dir.mkdir(parents=True, exist_ok=True)
        self.save_outputs = save_outputs

    def forward(self, return_f0_labels: bool = True, **inputs):
        f0_labels = None
        if "num_f0_bins" in self.task_names and return_f0_labels:
            f0_labels = self.get_f0_labels(inputs["mixture"])
            # Only count f0 bins during training and when counting is enabled
            if self.count_f0_bins and self.training:
                for f0_label in f0_labels:
                    label_val = f0_label.item()
                    self.f0_bin_counts[label_val] = (
                        self.f0_bin_counts.get(label_val, 0) + 1
                    )
        feats = self.feature_extractor(**inputs)
        return feats.view(feats.size(0), -1), f0_labels

    def training_step(self, batch, batch_idx):
        cue_features, cue_mask_ixs, loc_task_ixs, scene_features, labels = batch
        # Map task_name to correct label index
        targets = {}
        for task_name in self.task_names:
            if task_name == "num_word_classes":
                targets[task_name] = labels[:, 0].long().view(-1)
            elif task_name == "num_azim_classes":
                targets[task_name] = labels[:, 1].long().view(-1)
            elif task_name == "num_f0_bins":
                targets[task_name] = None
            else:
                raise ValueError(f"Unknown task_name: {task_name}")
        feats, f0_labels = self(mixture=scene_features, return_f0_labels=True)
        if "num_f0_bins" in self.task_names:
            targets["num_f0_bins"] = f0_labels

        for i, task_name in enumerate(self.task_names):
            optimizer = self.optimizers()[i]
            optimizer.zero_grad()
            head = self.classifier_heads[task_name]
            logits = head(feats)
            loss = F.cross_entropy(logits, targets[task_name])

            self.manual_backward(loss)
            optimizer.step()

            self.log(f"train_loss_{task_name}", loss, prog_bar=True)

    def validation_step(self, batch, batch_idx):
        cue_features, cue_mask_ixs, loc_task_ixs, scene_features, labels = batch
        targets = {}
        for task_name in self.task_names:
            if task_name == "num_word_classes":
                targets[task_name] = labels[:, 0].long().view(-1)
            elif task_name == "num_azim_classes":
                targets[task_name] = labels[:, 1].long().view(-1)
            elif task_name == "num_f0_bins":
                targets[task_name] = None
            else:
                raise ValueError(f"Unknown task_name: {task_name}")
        # Do not update f0_bin_counts in validation
        feats, f0_labels = self(mixture=scene_features, return_f0_labels=True)
        if "num_f0_bins" in self.task_names:
            targets["num_f0_bins"] = f0_labels

        for task_name in self.task_names:
            logits = self.classifier_heads[task_name](feats)

            loss = F.cross_entropy(logits, targets[task_name])
            self.log(f"val_loss_{task_name}", loss, prog_bar=True)

            # accumulate loss sum and counts per task for epoch-level averaging
            with torch.no_grad():
                total = targets[task_name].numel()
                self.val_loss_sum[task_name] += float(loss.item()) * int(total)
                self.val_total[task_name] += int(total)

    def on_train_epoch_end(self):
        # Save f0 bin counts after the first epoch and disable counting
        if self.current_epoch == 0 and self.count_f0_bins:
            self._save_f0_bin_counts()
            self.count_f0_bins = False  # Disable counting after first epoch

    def _save_f0_bin_counts(self):
        """Save f0 bin counts to a file after the first training epoch."""
        safe_stage = str(self.layer_idx).replace("/", "_")
        file_name = f"layer_{safe_stage}_f0_bin_counts_epoch_0.json"
        save_path = self.save_dir / file_name

        with open(save_path, "w") as f:
            json.dump(self.f0_bin_counts, f, indent=2)

        print(f"Saved f0 bin counts to {save_path}")

    def on_validation_epoch_end(self):
        # compute per-task average validation loss for the epoch, save best head if improved (lower is better)
        for task_name in self.task_names:
            total = self.val_total.get(task_name, 0)
            if total == 0:
                continue
            avg_loss = self.val_loss_sum[task_name] / float(total)
            self.log(
                f"val_avg_loss_{task_name}", avg_loss, prog_bar=True, sync_dist=True
            )

            if avg_loss < self.best_val_loss.get(task_name, float("inf")):
                self.best_val_loss[task_name] = avg_loss
                if self.save_outputs:
                    self._save_best_head_weights(task_name)

        # reset counters for next epoch
        for task_name in self.task_names:
            self.val_loss_sum[task_name] = 0.0
            self.val_total[task_name] = 0

    def _save_best_head_weights(self, task_name: str):
        head = self.classifier_heads[task_name]
        # file path: save under save_dir/layer_{ix}/{task_name}_best.pth
        try:
            layer_ix = int(self.layer_idx)
        except Exception:
            # fallback to target_stage text if layer_idx is unavailable
            layer_ix = str(self.layer_idx).replace("/", "_")
        save_subdir = self.save_dir / f"layer_{layer_ix}"
        save_subdir.mkdir(parents=True, exist_ok=True)
        file_name = f"{task_name}_best.pth"
        save_path = save_subdir / file_name
        torch.save(head.state_dict(), save_path)

    def configure_optimizers(self):
        return [
            torch.optim.Adam(
                self.classifier_heads[task].parameters(), lr=self.lr_dict[task]
            )
            for task in self.task_names
        ]

    def predict_step(self, batch, batch_idx):
        x = batch if isinstance(batch, torch.Tensor) else batch[0]
        feats, _ = self(mixture=x, return_f0_labels=False)
        return {
            task: torch.softmax(self.classifier_heads[task](feats), dim=1)
            for task in self.task_names
        }

    def get_f0_labels(self, batch):
        f0s = self.estimate_batch_f0(batch)
        f0_bins = torch.floor(12 * torch.log2(f0s / 50)).to(torch.int64)
        return f0_bins

    def estimate_batch_f0(self, batch):
        pitches = ta_F.detect_pitch_frequency(
            batch, sample_rate=44_100, freq_low=50, freq_high=300
        )
        pitches = pitches.mean(dim=(-2, -1))
        return pitches

In [4]:
def list_stage_names(model: nn.Module):
    """
    Print all available module names inside a model so you can pick a target stage.
    """
    print("Available stages in model:\n")
    for name, module in model.named_modules():
        print(f"{name:30s} ({module.__class__.__name__})")


def list_act_names(model: nn.Module):
    """
    Print all available module names inside a model so you can pick a target stage.
    """
    # print("Available stages in model:\n")
    name_list = []
    for name, module in model.named_modules():
        if "ReLU" in module.__class__.__name__:
            # print(f"{name:30s} ({module.__class__.__name__})")
            # get only the section with conv_block_X
            if "conv" in name:
                name = re.search(r"conv_block_[0-9]+", name).group(0)
            elif "relufc" in name:
                name = "relufc"
            name_list.append(name)
    return name_list


def format_stage_name(name: str):
    """
    Convert a stage name to a valid Python attribute name.
    """
    if "conv" in name:
        name = f"_orig_mod.model_dict.{name}.2"  # 2 is the ReLU layer after conv
    elif "fc" in name:
        name = "_orig_mod.relufc"
    return name


class ModelWithFrontEnd(nn.Module):
    def __init__(self, front_end, model):
        super().__init__()
        self.front_end = front_end
        self.model = model

    def forward(
        self,
        cue: torch.Tensor = None,
        mixture: torch.Tensor = None,
        cue_mask_ixs: torch.tensor = None,
    ):
        cue, mixture = self.front_end(cue, mixture)
        return self.model(cue, mixture, cue_mask_ixs)

In [5]:
layer_idx = 5
lr_word = 1e-3
lr_azim = 1e-3
lr_f0 = 1e-3
save_outputs = False
tasks = ["num_word_classes", "num_azim_classes", "num_f0_bins"]
config_path = "config/linear_readout/linear_readout_layer_5.yaml"

In [6]:
config_path = Path(config_path)
ckpt_path = Path(
    "attn_cue_models/word_task_v10_main_feature_gain_config/checkpoints/epoch=1-step=24679-v1.ckpt"
)

config = yaml.load(open(config_path, "r"), Loader=yaml.FullLoader)

config["num_workers"] = min(10, 32 // 2)
# config["hparas"]["batch_size"] = 64
# config["hparas"]["valid_step"] = 4_400
# config["corpus"]["root"] = "/orcd/data/jhm/001/datasets/dataset_binaural_attn/v10/"
# config["corpus"]["task"] = "word_and_location"
config["corpus"]["return_azim_loc_only"] = True
config["corpus"]["feature_gain"] = True
config["corpus"]["clean_percentage"] = 1.0

config["model"]["num_classes"]["num_locs"] = 72
# Step 1: Create model architecture
module = BinauralAttentionModule(config=config).cuda()

# Step 2: Load the checkpoint manually
checkpoint = torch.load(ckpt_path, map_location="cuda")

# Step 3: Load the weights only (ignore compilation and trainer states)
module.load_state_dict(checkpoint["state_dict"], strict=False)

cochgram = module.coch_gram
cnn = module.model
audio_transforms = module.audio_transforms
backbone = ModelWithFrontEnd(cochgram, cnn)

# 2. Choose a layer index for feature extraction
act_names = list_act_names(backbone)
if layer_idx >= len(act_names):
    raise ValueError(
        f"Layer index {layer_idx} out of range. Available layers: {len(act_names)}"
    )
layer_to_get = act_names[layer_idx]

# 3. Remove classifier layer to prevent interference
backbone.model.classification = nn.Identity()

# 4. Create full Lightning model
model = MultiClassifierModule(
    backbone=backbone,
    target_stage=layer_to_get,
    layer_idx=layer_idx,
    audio_transforms=audio_transforms,
    task_dict={
        "num_word_classes": 800,
        "num_azim_classes": 512,
        "num_f0_bins": 32,
    },
    lr_dict={
        "num_word_classes": lr_word,
        "num_azim_classes": lr_azim,
        "num_f0_bins": lr_f0,
    },
    save_outputs=save_outputs,
)

# WandB logger setup
if len(tasks) == 1:
    if tasks[0] == "num_word_classes":
        run_name = f"layer_{layer_idx}_word_lr_{lr_word}"
    elif tasks[0] == "num_azim_classes":
        run_name = f"layer_{layer_idx}_azim_lr_{lr_azim}"
    elif tasks[0] == "num_f0_bins":
        run_name = f"layer_{layer_idx}_f0_lr_{lr_f0}"
else:
    run_name = f"layer_{layer_idx}_multitask_lr_word_{lr_word}_lr_azim_{lr_azim}_lr_f0_{lr_f0}"
wandb_logger = WandbLogger(
    project="auditory_attn_linear_readout",
    name=run_name,
    group=f"layer_{layer_idx}",
)

Using explicit dim specification for demeaning in audio transforms
Batch in dataloader = False
Using BinauralAuditoryAttentionCNN
v08 True
num_classes={'num_words': 800, 'num_locs': 72}
Model performing both location and word tasks
Using singe gain function per layer
Conv block order: LN -> Conv -> ReLU
fc_attn: True
coch_affine: True
Using dataset BinauralAttentionDataset
BinauralAuditoryAttentionCNN(
  (model_dict): ModuleDict(
    (norm_coch_rep): LayerNorm((2, 40, 20000), eps=1e-05, elementwise_affine=True)
    (attn0): SimpleAttentionalGain()
    (conv_block_0): Sequential(
      (0): LayerNorm((2, 40, 20000), eps=1e-05, elementwise_affine=True)
      (1): Conv2d(2, 32, kernel_size=(2, 34), stride=(1, 1), bias=False)
      (2): ReLU()
    )
    (hann_pool_0): HannPooling2d()
    (attn1): SimpleAttentionalGain()
    (conv_block_1): Sequential(
      (0): LayerNorm((32, 20, 4992), eps=1e-05, elementwise_affine=True)
      (1): Conv2d(32, 64, kernel_size=(2, 14), stride=(1, 1), bias=

/orcd/data/jhm/001/om2/rphess/projects/github.com/Auditory-Attention/src/audio_transforms.py:338: UserWarning: "kaiser_window" resampling method name is being deprecated and replaced by "sinc_interp_kaiser" in the next release. The default behavior remains unchanged.
  self.downsampling_op = self.downsampling(self.sr,
/tmp/ipykernel_3014548/2713317060.py:22: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.seriali

In [7]:
# 5. Train
trainer = pl.Trainer(
    # detect_anomaly=True,
    accelerator="gpu",
    devices=1,
    max_epochs=2,
    val_check_interval=config["hparas"]["valid_step"],
    profiler=None,
    # strategy=DDPStrategy(find_unused_parameters=True),
    logger=wandb_logger,
)
trainer.fit(
    model,
    train_dataloaders=module.train_dataloader(),
    val_dataloaders=module.val_dataloader(),
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Using v06 dataset
/orcd/data/jhm/001/datasets/dataset_binaural_attn/v10/
Using 0.1 cue free data
Using gender balanced training 4M set
cue type: mixed
mixture_percentages={'voice_only': 0.5, 'voice_and_location': 0.5}
735 files in train concat dataset
len training set = 57330
Using v06 dataset
/orcd/data/jhm/001/datasets/dataset_binaural_attn/v10/
cue type: mixed
mixture_percentages={'voice_only': 0.5, 'voice_and_location': 0.5}
49 files in val concat dataset


wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: phess2 (phess2-massac) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


ServicePollForTokenError: Failed to read port info after 30.0 seconds.